
# 04 — Certified thick-handlebody ground-truth validation

This notebook tests whether the production pipeline can recover the spatial-graph topology of **genuinely thick embedded handlebodies**.

For each accepted ground-truth embedded graph \(G_i\),

\[
G_i
\longrightarrow
H_i
=
\{x:\operatorname{dist}(x,G_i)\le r_i\}
\longrightarrow
\text{certify the 3-D input topology}
\longrightarrow
\text{skeletonization}
\longrightarrow
G_i^{\rm rec}.
\]

## Crucial sampling rule

`N_GROUND_TRUTH` means the number of **valid, certified benchmark inputs**, not the number of attempted candidates.

A candidate is counted only if:

1. its continuous embedded graph is connected, bridgeless and subcubic;
2. its voxelized regular neighborhood has exactly the expected handlebody Betti numbers
   \[
   (\beta_0,\beta_1,\beta_2)=(1,g,0);
   \]
3. by default, its tube radius is at least `MIN_THICK_RADIUS_VOX` voxels, so thin under-resolved objects do not enter the benchmark.

Invalid candidates are rejected and new candidates are generated until exactly `N_GROUND_TRUTH` valid cases have been accepted.

## Reproducibility

Accepted embeddings are checkpointed to a compressed JSON archive every `CHECKPOINT_EVERY` accepted cases, and the final partial batch is always saved. The archive stores the full spatial graph (node positions and every edge polyline), the accepted radius, voxel spacing/origin, Betti certificate, graph metadata, and generation parameters. A CSV manifest is also saved.

The saved archive is sufficient to reconstruct and replot the exact accepted embeddings later without regenerating them.


Recovery results are likewise checkpointed every `CHECKPOINT_EVERY` tested cases to `synthetic_ground_truth_results_checkpoint.csv`.


## Crossing-rich stress test

The benchmark also enforces a quota of certified **crossing-rich** embeddings. For `N_GROUND_TRUTH = 1000` and the default `CROSSING_RICH_FRACTION = 0.60`, approximately 60% of the accepted random cases contain deliberately constructed over/under tubular crossings. A crossing-rich case is accepted only after the graph edges remain disjoint in 3-D, the tube surfaces have a safety gap, the reference projection verifies the crossings, and the complete thick volume passes the handlebody topology certificate.


In [1]:
from pathlib import Path


# ============================================================
# Benchmark configuration
# ============================================================

N_GROUND_TRUTH = 1000     # means 100 VALID certified inputs
CHECKPOINT_EVERY = 20       # save accepted embeddings/results every 20 cases
RANDOM_SEED = 20260824

GRID_SIZE = 200
THICKNESS_FRACTION = 0.90

# Random abstract ground-truth generation
MIN_NODES = 8
MAX_NODES = 18
MIN_CHORD_SPAN = 3
MIN_RANDOM_CHORDS = 2
WL_ITERATIONS = 5

# Embedding-family mix among ACCEPTED random cases
# For N_GROUND_TRUTH=1000 this makes about 60% of the random cases
# deliberately contain genuine over/under tubular crossings.
CROSSING_RICH_FRACTION = 0.60
MIN_DESIGNED_CROSSINGS = 2
MAX_DESIGNED_CROSSINGS = 12
CROSSING_HEIGHT = 0.32
CROSSING_ENDPOINT_MARGIN = 0.10
CROSSING_BUMP_MAX_WIDTH = 0.11
MIN_CROSSING_PARAMETER_SEPARATION = 0.055

# At every designed over/under event, centerline separation must remain
# larger than this many tube radii after the final thick radius is chosen.
# >2 guarantees disjoint tube surfaces; 2.4 adds a visible safety margin.
MIN_CROSSING_GAP_RADII = 2.40

# Verify the designed crossings with KnottedGraph's own XY projection code.
VERIFY_REFERENCE_PROJECTION_CROSSINGS = True

# Embedded geometry
EMBEDDING_RADIUS = 1.0
EDGE_SAMPLES = 81
LIFT_AMPLITUDE_1 = 0.11
LIFT_AMPLITUDE_2 = 0.055

# Conservative factor below the estimated first non-contact radius.
THICKNESS_SAFETY_FACTOR = 0.90

# "Valid" also means genuinely thick by default.
REQUIRE_MIN_THICK_RADIUS = True
MIN_THICK_RADIUS_VOX = 4.0

# Maximum number of random candidate attempts before giving a useful error.
# Increase this only if requesting a very large certified data set.
MAX_GENERATION_ATTEMPTS = max(10_000, 300 * N_GROUND_TRUTH)

# Production extraction / cleanup
MAX_JUNCTION_DEGREE = 3
ADAPTIVE_MAX_HOPS = 4
ANOMALY_RATIO = 0.15
CONTRACT_SHORT_EDGE_VOX = 1.75
SMOOTH_EPSILON_VOX = 1.5

# Yamada
COMPUTE_YAMADA = True
YAMADA_NUM_ROTATION_SAMPLES = 6
YAMADA_N_JOBS = 1

# Outputs
SAVE_RESULTS_CSV = True
RESULTS_CSV_NAME = "synthetic_ground_truth_yamada_preservation.csv"

SAVE_ACCEPTED_EMBEDDINGS = True
EMBEDDING_ARCHIVE_NAME = "certified_thick_handlebody_embeddings.json.gz"
EMBEDDING_MANIFEST_NAME = "certified_thick_handlebody_embeddings_manifest.csv"
RESULTS_CHECKPOINT_NAME = "synthetic_ground_truth_results_checkpoint.csv"

# Exact post-recovery graph archive.
#
# Every successful recovery stores the exact cleaned spatial graph used for
# recovered_yamada together with that polynomial and a SHA-256 fingerprint.
RECOVERED_GRAPH_ARCHIVE_NAME = "synthetic_ground_truth_recovered_graphs.json.gz"
RECOVERED_GRAPH_MANIFEST_NAME = "synthetic_ground_truth_recovered_graphs_manifest.csv"

# ------------------------------------------------------------------
# Paper-ready runtime measurements
# ------------------------------------------------------------------
TIMING_RESULTS_DIR = None
TIMING_RESULTS_CSV_NAME = "04_thick_handlebody_pipeline_timings.csv"
TIMING_SUMMARY_CSV_NAME = "04_thick_handlebody_pipeline_timing_summary.csv"

# Scientific recovery failures are data; do not abort the full run by default.
FAIL_ON_RECOVERY_FAILURE = False

assert N_GROUND_TRUTH >= 3
assert GRID_SIZE >= 32
assert 0.0 < THICKNESS_FRACTION < 1.0
assert MAX_NODES >= MIN_NODES
assert MIN_THICK_RADIUS_VOX > 0
assert CHECKPOINT_EVERY >= 1
assert 0.0 <= CROSSING_RICH_FRACTION <= 1.0
assert 1 <= MIN_DESIGNED_CROSSINGS <= MAX_DESIGNED_CROSSINGS
assert CROSSING_HEIGHT > 0
assert MIN_CROSSING_GAP_RADII > 2.0


In [2]:

from __future__ import annotations
import platform

from pathlib import Path
import itertools
import gzip
import hashlib
import json
import random
import subprocess
import sys
import time
import warnings
import math
import networkx as nx
import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import display

from scipy.ndimage import generate_binary_structure, label as ndi_label
from scipy.spatial import cKDTree
from skimage.measure import euler_number, marching_cubes

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Locate repository and import production KnottedGraph
# ------------------------------------------------------------

def _is_knotted_graph_repo(candidate: Path) -> bool:
    return (
        (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "knotted_graph").exists()
    )


ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if _is_knotted_graph_repo(candidate):
        ROOT = candidate
        break
    sibling = candidate / "KnottedGraph_1earlier"
    if _is_knotted_graph_repo(sibling):
        ROOT = sibling
        break
else:
    raise RuntimeError(
        "Run this notebook from inside the KnottedGraph repository checkout or FigureGeneration."
    )

BENCHMARK_DIR = ROOT / "User_guide" / "benchmarks"
BENCHMARK_RESULTS_DIR = BENCHMARK_DIR / "results"
HANDLEBODY_DATA_DIR = BENCHMARK_RESULTS_DIR / "handlebody_ground_truth"
if TIMING_RESULTS_DIR is None:
    TIMING_RESULTS_DIR = BENCHMARK_RESULTS_DIR
else:
    TIMING_RESULTS_DIR = Path(TIMING_RESULTS_DIR)

src_path = ROOT / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import knotted_graph
from knotted_graph.core import (
    contract_short_edges,
    ensure_embedding,
    remove_leaf_nodes,
    simplify_edges,
    smooth_edges,
)
from knotted_graph.extraction import (
    skeletonize_volume,
    topology_aware_skeleton_image_to_graph,
)
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.projection import compute_yamada_polynomial, select_projection

A = sp.Symbol("A")


def _git_text(*args: str) -> str:
    try:
        return subprocess.check_output(
            ["git", *args],
            cwd=ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except Exception:
        return "unknown"


print("Repository root:", ROOT)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Git branch:", _git_text("branch", "--show-current"))
print("Git commit:", _git_text("rev-parse", "--short", "HEAD"))
print("Native Yamada backend:", native_available())

if COMPUTE_YAMADA and not native_available():
    warnings.warn(
        f"Native Yamada backend unavailable: {native_import_error()}",
        RuntimeWarning,
    )

Repository root: /Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/KnottedGraph_1earlier
KnottedGraph: /Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/KnottedGraph_1earlier/src/knotted_graph/__init__.py
Git branch: integration/arbitrary-knot-fields-final-audit
Git commit: 62b4ac1
Native Yamada backend: True



## 1. Canonical embedding-sensitive controls


In [3]:

def cubic_bezier(p0, p1, p2, p3, n=181):
    p0, p1, p2, p3 = map(
        lambda x: np.asarray(x, dtype=float),
        (p0, p1, p2, p3),
    )
    t = np.linspace(0.0, 1.0, int(n))[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t ** 2 * p2
        + t ** 3 * p3
    )


def build_trivial_theta() -> nx.MultiGraph:
    g = nx.MultiGraph()
    left = np.array([-1.0, 0.0, 0.0])
    right = np.array([1.0, 0.0, 0.0])
    g.add_node(0, pos=left.copy())
    g.add_node(1, pos=right.copy())

    controls = [
        (
            np.array([-0.40, 0.90, 0.20]),
            np.array([ 0.40, 0.90,-0.20]),
        ),
        (
            np.array([-0.40,-0.90,-0.20]),
            np.array([ 0.40,-0.90, 0.20]),
        ),
        (
            np.array([-0.40, 0.00, 0.65]),
            np.array([ 0.40, 0.00, 0.65]),
        ),
    ]

    for c1, c2 in controls:
        g.add_edge(
            0,
            1,
            pts=cubic_bezier(left, c1, c2, right, n=181),
        )

    return ensure_embedding(g)


def _standard_trefoil(n=721, scale=0.42):
    t = np.linspace(0.0, 2.0 * np.pi, int(n))
    x = (2.0 + np.cos(3.0 * t)) * np.cos(2.0 * t)
    y = (2.0 + np.cos(3.0 * t)) * np.sin(2.0 * t)
    z = np.sin(3.0 * t)
    return scale * np.column_stack([x, y, z])


def build_trefoil_theta() -> nx.MultiGraph:
    knot = _standard_trefoil()
    split = (len(knot) - 1) // 2

    v0 = knot[0].copy()
    v1 = knot[split].copy()

    arc_a = knot[: split + 1].copy()
    arc_b = knot[split:][::-1].copy()

    c1 = v0 + np.array([0.00, -0.45, 2.10])
    c2 = v1 + np.array([0.00,  0.45, 2.10])
    arc_c = cubic_bezier(v0, c1, c2, v1, n=241)

    g = nx.MultiGraph()
    g.add_node(0, pos=v0.copy())
    g.add_node(1, pos=v1.copy())
    g.add_edge(0, 1, pts=arc_a)
    g.add_edge(0, 1, pts=arc_b)
    g.add_edge(0, 1, pts=arc_c)
    return ensure_embedding(g)


def build_k4_genus3() -> nx.MultiGraph:
    vertices = 0.72 * np.array(
        [
            [ 1.0, 1.0, 1.0],
            [ 1.0,-1.0,-1.0],
            [-1.0, 1.0,-1.0],
            [-1.0,-1.0, 1.0],
        ],
        dtype=float,
    )

    g = nx.MultiGraph()
    for node, point in enumerate(vertices):
        g.add_node(node, pos=point.copy())

    for u, v in itertools.combinations(range(4), 2):
        g.add_edge(
            u,
            v,
            pts=np.linspace(vertices[u], vertices[v], 121),
        )

    return ensure_embedding(g)



## 2. Random candidate generator: baseline + controlled crossings

The accepted random cases are drawn from two embedding families.

### Baseline family

A cycle plus a **noncrossing** chord matching is embedded by a smooth single-valued lift \(z=f(x,y)\). These are low-crossing controls.

### Crossing-rich family

A cycle plus a general chord matching is allowed to have projected chord intersections. Each intersection is converted into a genuine spatial over/under event:

\[
z_{\rm over}(t_c)=f(x_c,y_c)+h,
\qquad
z_{\rm under}(t_c)=f(x_c,y_c)-h.
\]

Compact smooth bumps are used, so the two graph edges remain disjoint in 3-D. These are **not graph vertices**.

A crossing-rich candidate is counted only if:

- it contains at least `MIN_DESIGNED_CROSSINGS` designed over/under events;
- the crossings are sufficiently separated along each edge;
- the final tube surfaces remain separated at every crossing;
- KnottedGraph's own reference XY projection sees the designed crossings;
- the thick voxelized object still has the correct handlebody Betti numbers.

The quota `CROSSING_RICH_FRACTION` is enforced on **accepted certified random cases**, not attempted candidates.


In [4]:

def _cyclic_distance(i: int, j: int, n: int) -> int:
    d = abs(j - i)
    return min(d, n - d)


def _chords_cross_on_circle(a: int, b: int, c: int, d: int) -> bool:
    a, b = sorted((a, b))
    c, d = sorted((c, d))
    return (a < c < b < d) or (c < a < d < b)


def _candidate_chord_pairs(n: int) -> list[tuple[int, int]]:
    return [
        (i, j)
        for i in range(n)
        for j in range(i + 1, n)
        if _cyclic_distance(i, j, n) >= MIN_CHORD_SPAN
    ]


def _random_matching(
    n: int,
    rng: random.Random,
    *,
    forbid_projected_crossings: bool,
) -> list[tuple[int, int]]:
    """Random matching on cycle vertices; each vertex gets at most one chord."""
    used: set[int] = set()
    chords: list[tuple[int, int]] = []

    candidates = _candidate_chord_pairs(n)
    rng.shuffle(candidates)
    keep_probability = rng.uniform(0.45, 1.00)

    for i, j in candidates:
        if i in used or j in used:
            continue

        if (
            forbid_projected_crossings
            and any(_chords_cross_on_circle(i, j, a, b) for a, b in chords)
        ):
            continue

        if rng.random() > keep_probability:
            continue

        chords.append((i, j))
        used.add(i)
        used.add(j)

    return sorted(chords)


def _random_noncrossing_matching(
    n: int,
    rng: random.Random,
) -> list[tuple[int, int]]:
    return _random_matching(
        n,
        rng,
        forbid_projected_crossings=True,
    )


def _lift_xy(points_xy: np.ndarray, phase: float) -> np.ndarray:
    points_xy = np.asarray(points_xy, dtype=float)
    x = points_xy[:, 0]
    y = points_xy[:, 1]

    z = (
        LIFT_AMPLITUDE_1 * np.sin(1.7 * x + 0.8 * y + phase)
        + LIFT_AMPLITUDE_2 * np.cos(1.1 * x - 1.3 * y + 0.5 * phase)
    )
    return np.column_stack([x, y, z])


def _circle_xy(n: int) -> np.ndarray:
    theta = 2.0 * np.pi * np.arange(n) / n
    return EMBEDDING_RADIUS * np.column_stack(
        [np.cos(theta), np.sin(theta)]
    )


def _segment_intersection_parameters(
    p0: np.ndarray,
    p1: np.ndarray,
    q0: np.ndarray,
    q1: np.ndarray,
):
    """
    Proper 2-D segment intersection.

    Returns (t, u, point) with
        p(t)=p0+t(p1-p0), q(u)=q0+u(q1-q0)
    for an interior/interior crossing, otherwise None.
    """
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    q0 = np.asarray(q0, dtype=float)
    q1 = np.asarray(q1, dtype=float)

    r = p1 - p0
    s = q1 - q0

    def cross2(a, b):
        return float(a[0] * b[1] - a[1] * b[0])

    denom = cross2(r, s)
    if abs(denom) <= 1e-12:
        return None

    qp = q0 - p0
    t = cross2(qp, s) / denom
    u = cross2(qp, r) / denom

    eps = 1e-9
    if not (eps < t < 1.0 - eps and eps < u < 1.0 - eps):
        return None

    point = p0 + t * r
    return float(t), float(u), point


def _chord_crossing_events(
    xy: np.ndarray,
    chords: list[tuple[int, int]],
) -> list[dict]:
    events = []

    for a_idx, (a, b) in enumerate(chords):
        for b_idx in range(a_idx + 1, len(chords)):
            c, d = chords[b_idx]

            # Matching chords have distinct endpoints by construction.
            result = _segment_intersection_parameters(
                xy[a], xy[b], xy[c], xy[d]
            )
            if result is None:
                continue

            t_a, t_b, point = result
            events.append(
                {
                    "edge_a_index": int(a_idx),
                    "edge_b_index": int(b_idx),
                    "t_a": float(t_a),
                    "t_b": float(t_b),
                    "xy": [float(point[0]), float(point[1])],
                }
            )

    return events


def _crossing_parameters_are_well_separated(
    events: list[dict],
    n_chords: int,
) -> bool:
    params = {i: [] for i in range(n_chords)}

    for event in events:
        params[event["edge_a_index"]].append(event["t_a"])
        params[event["edge_b_index"]].append(event["t_b"])

    for values in params.values():
        values = sorted(values)

        for t in values:
            if (
                t < CROSSING_ENDPOINT_MARGIN
                or t > 1.0 - CROSSING_ENDPOINT_MARGIN
            ):
                return False

        if len(values) > 1:
            gaps = np.diff(values)
            if float(np.min(gaps)) < MIN_CROSSING_PARAMETER_SEPARATION:
                return False

    return True


def _compact_bump(t: np.ndarray, center: float, width: float) -> np.ndarray:
    """C1 compact bump: 1 at center and 0 outside |t-center|<width."""
    x = (t - center) / width
    out = np.zeros_like(t, dtype=float)
    inside = np.abs(x) < 1.0
    out[inside] = np.cos(0.5 * np.pi * x[inside]) ** 2
    return out


def _bump_widths_for_events(
    events: list[dict],
    n_chords: int,
) -> dict[tuple[int, int], float]:
    """
    Pick nonoverlapping compact-support widths along every crossing chord.
    Key is (event_index, chord_index).
    """
    by_chord: dict[int, list[tuple[int, float]]] = {
        i: [] for i in range(n_chords)
    }

    for event_index, event in enumerate(events):
        by_chord[event["edge_a_index"]].append(
            (event_index, event["t_a"])
        )
        by_chord[event["edge_b_index"]].append(
            (event_index, event["t_b"])
        )

    widths: dict[tuple[int, int], float] = {}

    for chord_index, entries in by_chord.items():
        entries = sorted(entries, key=lambda item: item[1])

        for local_index, (event_index, center) in enumerate(entries):
            nearby = [
                center,
                1.0 - center,
            ]

            if local_index > 0:
                nearby.append(center - entries[local_index - 1][1])
            if local_index + 1 < len(entries):
                nearby.append(entries[local_index + 1][1] - center)

            nearest = min(nearby)
            width = min(
                CROSSING_BUMP_MAX_WIDTH,
                0.42 * nearest,
            )
            width = max(width, 0.018)
            widths[(event_index, chord_index)] = float(width)

    return widths


def embed_cycle_chord_graph(
    simple_graph: nx.Graph,
    phase: float,
) -> nx.MultiGraph:
    """Baseline low-crossing smooth lift."""
    n = simple_graph.number_of_nodes()
    xy = _circle_xy(n)
    xyz = _lift_xy(xy, phase)

    embedded = nx.MultiGraph()

    for node in simple_graph.nodes:
        embedded.add_node(node, pos=xyz[node].copy())

    for u, v in simple_graph.edges:
        edge_xy = np.linspace(xy[u], xy[v], EDGE_SAMPLES)
        edge_xyz = _lift_xy(edge_xy, phase)
        edge_xyz[0] = xyz[u]
        edge_xyz[-1] = xyz[v]
        embedded.add_edge(u, v, pts=edge_xyz)

    return ensure_embedding(embedded)


def embed_crossing_rich_cycle_chord_graph(
    n: int,
    chords: list[tuple[int, int]],
    phase: float,
    rng: random.Random,
) -> tuple[nx.MultiGraph, list[dict]]:
    """
    Embed a cycle plus chord matching with controlled spatial over/under crossings.

    Cycle edges retain the smooth lift. Chords receive compact ±z bridge bumps
    at every projected chord crossing.
    """
    xy = _circle_xy(n)
    xyz = _lift_xy(xy, phase)

    events = _chord_crossing_events(xy, chords)

    if not (
        MIN_DESIGNED_CROSSINGS
        <= len(events)
        <= MAX_DESIGNED_CROSSINGS
    ):
        raise ValueError(
            f"crossing count {len(events)} outside "
            f"[{MIN_DESIGNED_CROSSINGS},{MAX_DESIGNED_CROSSINGS}]"
        )

    if not _crossing_parameters_are_well_separated(events, len(chords)):
        raise ValueError("crossings too close to endpoints or to each other")

    widths = _bump_widths_for_events(events, len(chords))

    # Random over/under assignment at each designed crossing.
    per_chord: dict[int, list[tuple[int, float, float, int]]] = {
        i: [] for i in range(len(chords))
    }

    for event_index, event in enumerate(events):
        a = event["edge_a_index"]
        b = event["edge_b_index"]

        if rng.random() < 0.5:
            over, under = a, b
        else:
            over, under = b, a

        event["over_chord_index"] = int(over)
        event["under_chord_index"] = int(under)
        event["designed_centerline_gap_world"] = float(
            2.0 * CROSSING_HEIGHT
        )

        t_for = {
            a: event["t_a"],
            b: event["t_b"],
        }

        per_chord[over].append(
            (
                event_index,
                float(t_for[over]),
                +1.0,
                int(over),
            )
        )
        per_chord[under].append(
            (
                event_index,
                float(t_for[under]),
                -1.0,
                int(under),
            )
        )

    embedded = nx.MultiGraph()

    for node in range(n):
        embedded.add_node(node, pos=xyz[node].copy())

    # Cycle edges.
    for u in range(n):
        v = (u + 1) % n
        edge_xy = np.linspace(xy[u], xy[v], EDGE_SAMPLES)
        edge_xyz = _lift_xy(edge_xy, phase)
        edge_xyz[0] = xyz[u]
        edge_xyz[-1] = xyz[v]
        embedded.add_edge(
            u,
            v,
            pts=edge_xyz,
            role="cycle",
        )

    # Crossing chords.
    t = np.linspace(0.0, 1.0, EDGE_SAMPLES)

    for chord_index, (u, v) in enumerate(chords):
        edge_xy = (
            (1.0 - t)[:, None] * xy[u]
            + t[:, None] * xy[v]
        )
        edge_xyz = _lift_xy(edge_xy, phase)

        for event_index, center, sign, _ in per_chord[chord_index]:
            width = widths[(event_index, chord_index)]
            edge_xyz[:, 2] += (
                sign
                * CROSSING_HEIGHT
                * _compact_bump(t, center, width)
            )

        edge_xyz[0] = xyz[u]
        edge_xyz[-1] = xyz[v]

        embedded.add_edge(
            u,
            v,
            pts=edge_xyz,
            role="chord",
            chord_index=int(chord_index),
        )

    # Record actual z values at the designed XY crossings from the analytic
    # construction. Compact supports are non-overlapping, so the intended
    # centerline gap is exactly 2*CROSSING_HEIGHT up to interpolation.
    return ensure_embedding(embedded), events


def _build_simple_cycle_chord_graph(
    n: int,
    chords: list[tuple[int, int]],
) -> nx.Graph:
    graph = nx.cycle_graph(n)
    graph.add_edges_from(chords)
    return graph


def make_random_candidate(
    rng: random.Random,
    *,
    embedding_family: str,
) -> dict:
    """
    Generate one abstract+embedded candidate from the requested family.
    Certification and accepted-family quotas happen later.
    """
    n = rng.randint(MIN_NODES, MAX_NODES)
    phase = rng.uniform(0.0, 2.0 * np.pi)

    if embedding_family == "baseline":
        chords = _random_noncrossing_matching(n, rng)

        if len(chords) < MIN_RANDOM_CHORDS:
            return {
                "accepted_as_candidate": False,
                "reason": "too_few_chords",
            }

        graph = _build_simple_cycle_chord_graph(n, chords)
        embedded = embed_cycle_chord_graph(graph, phase)
        crossing_events = []

    elif embedding_family == "crossing_rich":
        # A few local matching attempts make it inexpensive to obtain
        # 2-8 well-spaced projected chord crossings.
        chords = None
        crossing_events = None
        embedded = None
        graph = None

        for _ in range(40):
            trial = _random_matching(
                n,
                rng,
                forbid_projected_crossings=False,
            )

            if len(trial) < max(MIN_RANDOM_CHORDS, 3):
                continue

            trial_graph = _build_simple_cycle_chord_graph(n, trial)
            xy = _circle_xy(n)
            events = _chord_crossing_events(xy, trial)

            if not (
                MIN_DESIGNED_CROSSINGS
                <= len(events)
                <= MAX_DESIGNED_CROSSINGS
            ):
                continue

            if not _crossing_parameters_are_well_separated(
                events,
                len(trial),
            ):
                continue

            try:
                trial_embedded, events = (
                    embed_crossing_rich_cycle_chord_graph(
                        n,
                        trial,
                        phase,
                        rng,
                    )
                )
            except ValueError:
                continue

            chords = trial
            graph = trial_graph
            embedded = trial_embedded
            crossing_events = events
            break

        if embedded is None:
            return {
                "accepted_as_candidate": False,
                "reason": "could_not_construct_crossing_rich_embedding",
            }

    else:
        raise ValueError(
            "embedding_family must be 'baseline' or 'crossing_rich'"
        )

    if not nx.is_connected(graph):
        return {
            "accepted_as_candidate": False,
            "reason": "disconnected",
        }

    if max(dict(graph.degree()).values()) > 3:
        return {
            "accepted_as_candidate": False,
            "reason": "degree_gt_3",
        }

    if list(nx.bridges(graph)):
        return {
            "accepted_as_candidate": False,
            "reason": "has_bridge",
        }

    wl_hash = nx.weisfeiler_lehman_graph_hash(
        graph,
        iterations=WL_ITERATIONS,
    )

    min_crossing_gap = (
        min(
            event["designed_centerline_gap_world"]
            for event in crossing_events
        )
        if crossing_events
        else None
    )

    return {
        "accepted_as_candidate": True,
        "kind": "random_cycle_chord",
        "embedding_family": embedding_family,
        "graph": embedded,
        "wl_hash": wl_hash,
        "n_cycle_nodes": int(n),
        "n_chords": int(len(chords)),
        "phase": float(phase),
        "chords": [tuple(map(int, chord)) for chord in chords],
        "designed_crossings": int(len(crossing_events)),
        "crossing_events": crossing_events,
        "min_designed_crossing_gap_world": (
            None
            if min_crossing_gap is None
            else float(min_crossing_gap)
        ),
    }



## 3. Graph-level statistics used by certification


In [5]:

def graph_cycle_rank(graph: nx.MultiGraph) -> int:
    if graph.number_of_nodes() == 0:
        return 0

    return int(
        graph.number_of_edges()
        - graph.number_of_nodes()
        + nx.number_connected_components(graph)
    )


def graph_stats(graph: nx.MultiGraph) -> dict:
    if graph.number_of_nodes() == 0:
        return {
            "nodes": 0,
            "edges": 0,
            "components": 0,
            "cycle_rank": 0,
            "max_degree": 0,
            "connected": False,
            "subcubic": False,
        }

    degrees = dict(graph.degree())
    components = nx.number_connected_components(graph)

    return {
        "nodes": int(graph.number_of_nodes()),
        "edges": int(graph.number_of_edges()),
        "components": int(components),
        "cycle_rank": int(
            graph.number_of_edges()
            - graph.number_of_nodes()
            + components
        ),
        "max_degree": int(max(degrees.values(), default=0)),
        "connected": bool(components == 1),
        "subcubic": bool(max(degrees.values(), default=0) <= 3),
    }



## 4. Estimate the largest safe tube radius

For each embedded spine, we estimate a conservative centerline-separation scale. Intentional meetings at common graph vertices are excluded.

The actual radius is then

\[
r_i
=
\texttt{THICKNESS\_FRACTION}
\times
r_{i,\mathrm{limit}},
\]

with one fixed value

\[
\texttt{THICKNESS\_FRACTION}=0.90.
\]

Thus every case is tested close to its own geometric non-contact limit rather than with a thin universal voxel radius.


In [6]:

def _resample_polyline(points: np.ndarray, n: int = 161) -> np.ndarray:
    points = np.asarray(points, dtype=float)

    segment_lengths = np.linalg.norm(
        np.diff(points, axis=0),
        axis=1,
    )
    arclength = np.concatenate(
        [[0.0], np.cumsum(segment_lengths)]
    )

    if arclength[-1] <= 1e-15:
        return np.repeat(points[:1], int(n), axis=0)

    query = np.linspace(
        0.0,
        arclength[-1],
        int(n),
    )

    return np.column_stack(
        [
            np.interp(query, arclength, points[:, axis])
            for axis in range(3)
        ]
    )


def estimate_thickness_limit(
    graph: nx.MultiGraph,
    *,
    endpoint_exclusion_fraction: float = 0.14,
):
    edges = []

    for u, v, key, data in graph.edges(keys=True, data=True):
        points = _resample_polyline(
            np.asarray(data["pts"], dtype=float),
            n=161,
        )

        cut = max(
            2,
            int(endpoint_exclusion_fraction * (len(points) - 1)),
        )

        interior = (
            points[cut:-cut]
            if 2 * cut < len(points)
            else points
        )

        edges.append(
            {
                "u": u,
                "v": v,
                "full": points,
                "interior": interior,
            }
        )

    minimum_clearance = np.inf

    for i, edge_i in enumerate(edges):
        for edge_j in edges[i + 1:]:
            share_vertex = bool(
                {edge_i["u"], edge_i["v"]}
                & {edge_j["u"], edge_j["v"]}
            )

            points_i = (
                edge_i["interior"]
                if share_vertex
                else edge_i["full"]
            )
            points_j = (
                edge_j["interior"]
                if share_vertex
                else edge_j["full"]
            )

            distance, _ = cKDTree(points_j).query(
                points_i,
                k=1,
            )
            minimum_clearance = min(
                minimum_clearance,
                float(np.min(distance)),
            )

    if not np.isfinite(minimum_clearance):
        raise RuntimeError(
            "Could not estimate a finite non-contact clearance."
        )

    return (
        0.5
        * THICKNESS_SAFETY_FACTOR
        * minimum_clearance
    )



## 5. Build the thick 3-D regular-neighborhood volume

The volume is evaluated directly against the continuous polyline segments. The seed graph is not passed to the recovery stage.


In [7]:

def all_edge_points(graph: nx.MultiGraph) -> np.ndarray:
    return np.concatenate(
        [
            np.asarray(data["pts"], dtype=float)
            for _, _, _, data in graph.edges(
                keys=True,
                data=True,
            )
        ],
        axis=0,
    )


def voxelize_regular_neighborhood(
    graph: nx.MultiGraph,
    radius: float,
    grid_size: int,
    *,
    margin_radii: float = 2.25,
):
    points = all_edge_points(graph)

    max_norm = float(
        np.max(
            np.linalg.norm(points, axis=1)
        )
    )

    half_extent = max_norm + margin_radii * radius

    origin = np.array(
        [-half_extent, -half_extent, -half_extent],
        dtype=float,
    )

    spacing = (
        2.0 * half_extent / (grid_size - 1)
    )

    volume = np.zeros(
        (grid_size, grid_size, grid_size),
        dtype=bool,
    )

    for _, _, _, data in graph.edges(
        keys=True,
        data=True,
    ):
        polyline = np.asarray(
            data["pts"],
            dtype=float,
        )

        for a, b in zip(
            polyline[:-1],
            polyline[1:],
        ):
            lower = np.floor(
                (
                    np.minimum(a, b)
                    - radius
                    - origin
                )
                / spacing
            ).astype(int)

            upper = np.ceil(
                (
                    np.maximum(a, b)
                    + radius
                    - origin
                )
                / spacing
            ).astype(int)

            lower = np.maximum(lower, 0)
            upper = np.minimum(
                upper,
                grid_size - 1,
            )

            if np.any(lower > upper):
                continue

            sx = slice(
                lower[0],
                upper[0] + 1,
            )
            sy = slice(
                lower[1],
                upper[1] + 1,
            )
            sz = slice(
                lower[2],
                upper[2] + 1,
            )

            x = (
                origin[0]
                + spacing
                * np.arange(
                    lower[0],
                    upper[0] + 1,
                )[:, None, None]
            )
            y = (
                origin[1]
                + spacing
                * np.arange(
                    lower[1],
                    upper[1] + 1,
                )[None, :, None]
            )
            z = (
                origin[2]
                + spacing
                * np.arange(
                    lower[2],
                    upper[2] + 1,
                )[None, None, :]
            )

            ab = b - a
            denominator = float(ab @ ab)

            if denominator <= 1e-20:
                distance_squared = (
                    (x - a[0]) ** 2
                    + (y - a[1]) ** 2
                    + (z - a[2]) ** 2
                )
            else:
                t = (
                    (x - a[0]) * ab[0]
                    + (y - a[1]) * ab[1]
                    + (z - a[2]) * ab[2]
                ) / denominator

                t = np.clip(
                    t,
                    0.0,
                    1.0,
                )

                cx = a[0] + t * ab[0]
                cy = a[1] + t * ab[1]
                cz = a[2] + t * ab[2]

                distance_squared = (
                    (x - cx) ** 2
                    + (y - cy) ** 2
                    + (z - cz) ** 2
                )

            volume[sx, sy, sz] |= (
                distance_squared
                <= radius * radius
            )

    return (
        volume,
        origin,
        float(spacing),
        float(half_extent),
    )



## 6. Independent input-topology certificate

For a connected genus-\(g\) handlebody,

\[
(\beta_0,\beta_1,\beta_2)=(1,g,0).
\]

The generator below **rejects** any candidate whose voxelized input does not satisfy this certificate. Rejected candidates never enter `CASES` and therefore never count toward `N_GROUND_TRUTH`.


In [8]:

def volume_betti(volume: np.ndarray) -> dict:
    mask = np.asarray(
        volume,
        dtype=bool,
    )

    _, beta0 = ndi_label(
        mask,
        structure=generate_binary_structure(3, 3),
    )

    padded = np.pad(
        mask,
        1,
        mode="constant",
        constant_values=False,
    )

    _, background_components = ndi_label(
        ~padded,
        structure=generate_binary_structure(3, 1),
    )

    beta2 = max(
        0,
        int(background_components) - 1,
    )

    chi = int(
        euler_number(
            mask,
            connectivity=3,
        )
    )

    beta1 = int(
        beta0 + beta2 - chi
    )

    return {
        "beta0": int(beta0),
        "beta1": int(beta1),
        "beta2": int(beta2),
        "chi": int(chi),
    }


def expected_handlebody_betti(
    graph: nx.MultiGraph,
) -> dict:
    genus = graph_cycle_rank(graph)

    return {
        "beta0": 1,
        "beta1": genus,
        "beta2": 0,
        "chi": 1 - genus,
    }


# ============================================================
# Certified-case generation
# ============================================================

def _planned_voxel_geometry(
    graph: nx.MultiGraph,
    radius: float,
    grid_size: int,
    *,
    margin_radii: float = 2.25,
) -> tuple[np.ndarray, float, float, float]:
    """Return origin, spacing, half_extent, radius_vox without allocating the volume."""
    points = all_edge_points(graph)
    max_norm = float(np.max(np.linalg.norm(points, axis=1)))
    half_extent = max_norm + margin_radii * radius
    origin = np.array([-half_extent, -half_extent, -half_extent], dtype=float)
    spacing = 2.0 * half_extent / (grid_size - 1)
    radius_vox = radius / spacing
    return origin, float(spacing), float(half_extent), float(radius_vox)


def multigraph_has_bridge(graph: nx.MultiGraph) -> bool:
    """Return True if removing any single multiedge disconnects the graph."""
    if graph.number_of_edges() == 0:
        return False

    base_components = nx.number_connected_components(graph)

    for u, v, key in list(graph.edges(keys=True)):
        test = graph.copy()
        test.remove_edge(u, v, key)
        if nx.number_connected_components(test) > base_components:
            return True

    return False


def certify_candidate(case: dict) -> tuple[bool, dict]:
    """
    Certify a candidate BEFORE it is admitted into CASES.

    Fast rejection happens first using radius_vox. Only sufficiently thick
    candidates allocate a GRID_SIZE^3 volume and undergo Betti certification.
    """
    graph = case["graph"]
    stats = graph_stats(graph)

    if not stats["connected"]:
        return False, {"reject_reason": "graph_disconnected"}
    if not stats["subcubic"]:
        return False, {"reject_reason": "graph_not_subcubic"}
    if multigraph_has_bridge(graph):
        return False, {"reject_reason": "graph_has_bridge"}

    radius_limit = estimate_thickness_limit(graph)
    radius = THICKNESS_FRACTION * radius_limit

    planned_origin, planned_spacing, planned_half_extent, radius_vox = (
        _planned_voxel_geometry(
            graph,
            radius,
            GRID_SIZE,
        )
    )

    if REQUIRE_MIN_THICK_RADIUS and radius_vox < MIN_THICK_RADIUS_VOX:
        return False, {
            "reject_reason": "tube_under_resolved",
            "radius_limit_world": float(radius_limit),
            "radius_world": float(radius),
            "spacing_world": float(planned_spacing),
            "radius_vox": float(radius_vox),
        }

    reference_projection_crossings = None

    if case.get("embedding_family") == "crossing_rich":
        designed_crossings = int(case.get("designed_crossings", 0))
        min_gap = case.get("min_designed_crossing_gap_world")

        if designed_crossings < MIN_DESIGNED_CROSSINGS:
            return False, {
                "reject_reason": "too_few_designed_crossings",
                "designed_crossings": designed_crossings,
            }

        if min_gap is None or float(min_gap) < MIN_CROSSING_GAP_RADII * radius:
            return False, {
                "reject_reason": "crossing_tubes_too_close",
                "designed_crossings": designed_crossings,
                "min_designed_crossing_gap_world": min_gap,
                "required_gap_world": float(MIN_CROSSING_GAP_RADII * radius),
                "radius_world": float(radius),
                "radius_vox": float(radius_vox),
            }

        if VERIFY_REFERENCE_PROJECTION_CROSSINGS:
            try:
                projection = select_projection(
                    graph,
                    rotation_angles=(0.0, 0.0, 0.0),
                )
                reference_projection_crossings = int(
                    projection.num_crossings
                )
            except Exception as exc:
                return False, {
                    "reject_reason": "reference_projection_failed",
                    "projection_error": f"{type(exc).__name__}: {exc}",
                }

            if reference_projection_crossings < designed_crossings:
                return False, {
                    "reject_reason": "designed_crossings_not_verified",
                    "designed_crossings": designed_crossings,
                    "reference_projection_crossings": (
                        reference_projection_crossings
                    ),
                }

    volume, origin, spacing, half_extent = voxelize_regular_neighborhood(
        graph,
        radius,
        GRID_SIZE,
    )

    # These should agree exactly up to floating-point arithmetic.
    if not np.allclose(origin, planned_origin):
        raise RuntimeError("Planned and actual voxel origins disagree.")
    if not np.isclose(spacing, planned_spacing):
        raise RuntimeError("Planned and actual voxel spacings disagree.")

    observed = volume_betti(volume)
    expected = expected_handlebody_betti(graph)
    topology_pass = observed == expected

    certification = {
        "reject_reason": None if topology_pass else "input_topology_certificate_failed",
        "radius_limit_world": float(radius_limit),
        "radius_world": float(radius),
        "origin_world": [float(x) for x in origin],
        "spacing_world": float(spacing),
        "half_extent_world": float(half_extent),
        "radius_vox": float(radius_vox),
        "occupied_voxels": int(np.count_nonzero(volume)),
        "occupancy_fraction": float(np.mean(volume)),
        "expected_beta0": int(expected["beta0"]),
        "expected_beta1": int(expected["beta1"]),
        "expected_beta2": int(expected["beta2"]),
        "expected_chi": int(expected["chi"]),
        "observed_beta0": int(observed["beta0"]),
        "observed_beta1": int(observed["beta1"]),
        "observed_beta2": int(observed["beta2"]),
        "observed_chi": int(observed["chi"]),
        "input_topology_pass": bool(topology_pass),
        "thick_enough": bool(radius_vox >= MIN_THICK_RADIUS_VOX),
        "embedding_family": case.get("embedding_family", "control"),
        "designed_crossings": int(case.get("designed_crossings", 0)),
        "reference_projection_crossings": reference_projection_crossings,
        "min_designed_crossing_gap_world": case.get(
            "min_designed_crossing_gap_world"
        ),
        "crossing_gap_in_tube_radii": (
            None
            if case.get("min_designed_crossing_gap_world") is None
            else float(case["min_designed_crossing_gap_world"]) / radius
        ),
    }

    # Do not retain the 200^3 volume for every accepted case.
    del volume

    return bool(topology_pass), certification


def _json_node_id(value):
    """Convert the integer/string node IDs used here to JSON-native scalars."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def graph_to_serializable(graph: nx.MultiGraph) -> dict:
    """Store the full embedded graph needed to reproduce/plot it later."""
    graph = ensure_embedding(graph, copy=True, normalize=True)

    nodes = [
        {
            "id": _json_node_id(node),
            "pos": np.asarray(data["pos"], dtype=float).tolist(),
        }
        for node, data in graph.nodes(data=True)
    ]

    edges = [
        {
            "u": _json_node_id(u),
            "v": _json_node_id(v),
            "key": _json_node_id(key),
            "pts": np.asarray(data["pts"], dtype=float).tolist(),
        }
        for u, v, key, data in graph.edges(keys=True, data=True)
    ]

    return {"nodes": nodes, "edges": edges}


def graph_from_serializable(payload: dict) -> nx.MultiGraph:
    """Reconstruct one exact embedded MultiGraph from the saved archive."""
    graph = nx.MultiGraph()

    for node in payload["nodes"]:
        graph.add_node(
            node["id"],
            pos=np.asarray(node["pos"], dtype=float),
        )

    for edge in payload["edges"]:
        graph.add_edge(
            edge["u"],
            edge["v"],
            key=edge["key"],
            pts=np.asarray(edge["pts"], dtype=float),
        )

    return ensure_embedding(graph)


def save_embedding_archive(cases: list[dict]) -> tuple[Path, Path]:
    """
    Save all accepted embeddings and the full information needed to
    reconstruct their thick volumes later.
    """
    archive_path = HANDLEBODY_DATA_DIR / EMBEDDING_ARCHIVE_NAME
    manifest_path = HANDLEBODY_DATA_DIR / EMBEDDING_MANIFEST_NAME

    archive = {
        "schema_version": 1,
        "description": "Certified thick-handlebody ground-truth embeddings",
        "config": {
            "N_GROUND_TRUTH": int(N_GROUND_TRUTH),
            "RANDOM_SEED": int(RANDOM_SEED),
            "GRID_SIZE": int(GRID_SIZE),
            "THICKNESS_FRACTION": float(THICKNESS_FRACTION),
            "REQUIRE_MIN_THICK_RADIUS": bool(REQUIRE_MIN_THICK_RADIUS),
            "MIN_THICK_RADIUS_VOX": float(MIN_THICK_RADIUS_VOX),
            "MIN_NODES": int(MIN_NODES),
            "MAX_NODES": int(MAX_NODES),
            "MIN_CHORD_SPAN": int(MIN_CHORD_SPAN),
            "MIN_RANDOM_CHORDS": int(MIN_RANDOM_CHORDS),
            "WL_ITERATIONS": int(WL_ITERATIONS),
            "CROSSING_RICH_FRACTION": float(CROSSING_RICH_FRACTION),
            "MIN_DESIGNED_CROSSINGS": int(MIN_DESIGNED_CROSSINGS),
            "MAX_DESIGNED_CROSSINGS": int(MAX_DESIGNED_CROSSINGS),
            "CROSSING_HEIGHT": float(CROSSING_HEIGHT),
            "MIN_CROSSING_GAP_RADII": float(MIN_CROSSING_GAP_RADII),
            "EMBEDDING_RADIUS": float(EMBEDDING_RADIUS),
            "EDGE_SAMPLES": int(EDGE_SAMPLES),
            "LIFT_AMPLITUDE_1": float(LIFT_AMPLITUDE_1),
            "LIFT_AMPLITUDE_2": float(LIFT_AMPLITUDE_2),
            "THICKNESS_SAFETY_FACTOR": float(THICKNESS_SAFETY_FACTOR),
        },
        "cases": [],
    }

    manifest_rows = []

    for case in cases:
        stats = graph_stats(case["graph"])
        cert = case["certification"]

        record = {
            "case": case["case"],
            "kind": case["kind"],
            "wl_hash": case.get("wl_hash"),
            "n_cycle_nodes": case.get("n_cycle_nodes"),
            "n_chords": case.get("n_chords"),
            "phase": case.get("phase"),
            "chords": case.get("chords"),
            "embedding_family": case.get("embedding_family", "control"),
            "designed_crossings": int(case.get("designed_crossings", 0)),
            "crossing_events": case.get("crossing_events", []),
            "min_designed_crossing_gap_world": case.get(
                "min_designed_crossing_gap_world"
            ),
            "graph_stats": stats,
            "certification": cert,
            "embedding": graph_to_serializable(case["graph"]),
        }
        archive["cases"].append(record)

        manifest_rows.append(
            {
                "case": case["case"],
                "kind": case["kind"],
                "embedding_family": case.get("embedding_family", "control"),
                "wl_hash": case.get("wl_hash"),
                "designed_crossings": int(case.get("designed_crossings", 0)),
                "reference_projection_crossings": cert.get(
                    "reference_projection_crossings"
                ),
                "crossing_gap_in_tube_radii": cert.get(
                    "crossing_gap_in_tube_radii"
                ),
                "nodes": stats["nodes"],
                "edges": stats["edges"],
                "genus_cycle_rank": stats["cycle_rank"],
                "max_degree": stats["max_degree"],
                "radius_world": cert["radius_world"],
                "radius_vox": cert["radius_vox"],
                "spacing_world": cert["spacing_world"],
                "input_topology_pass": cert["input_topology_pass"],
                "thick_enough": cert["thick_enough"],
                "beta0": cert["observed_beta0"],
                "beta1": cert["observed_beta1"],
                "beta2": cert["observed_beta2"],
                "chi": cert["observed_chi"],
            }
        )

    # Atomic checkpoint writes: write temporary files first, then replace.
    archive_tmp = archive_path.with_suffix(archive_path.suffix + ".tmp")
    manifest_tmp = manifest_path.with_suffix(manifest_path.suffix + ".tmp")

    with gzip.open(archive_tmp, "wt", encoding="utf-8") as handle:
        json.dump(
            archive,
            handle,
            separators=(",", ":"),
            allow_nan=False,
        )

    pd.DataFrame(manifest_rows).to_csv(
        manifest_tmp,
        index=False,
    )

    archive_tmp.replace(archive_path)
    manifest_tmp.replace(manifest_path)

    return archive_path, manifest_path


def load_embedding_archive(
    archive_path: str | Path = EMBEDDING_ARCHIVE_NAME,
) -> dict:
    """Load a saved archive. Relative paths are interpreted from ROOT."""
    archive_path = Path(archive_path)
    if not archive_path.is_absolute():
        archive_path = ROOT / archive_path

    with gzip.open(archive_path, "rt", encoding="utf-8") as handle:
        return json.load(handle)


def reconstruct_saved_case(
    case_name: str,
    archive_path: str | Path = EMBEDDING_ARCHIVE_NAME,
) -> tuple[nx.MultiGraph, dict]:
    """Return (embedded_graph, saved_record) for later plotting/reanalysis."""
    archive = load_embedding_archive(archive_path)

    for record in archive["cases"]:
        if record["case"] == case_name:
            return graph_from_serializable(record["embedding"]), record

    raise KeyError(f"Case {case_name!r} not found in archive.")


def _control_cases() -> list[dict]:
    return [
        {
            "case": "control_trivial_theta",
            "kind": "control",
            "embedding_family": "baseline_control",
            "graph": build_trivial_theta(),
            "wl_hash": None,
            "n_cycle_nodes": None,
            "n_chords": None,
            "phase": None,
            "chords": None,
        },
        {
            "case": "control_trefoil_theta",
            "kind": "control",
            "embedding_family": "knotted_crossing_control",
            "graph": build_trefoil_theta(),
            "wl_hash": None,
            "n_cycle_nodes": None,
            "n_chords": None,
            "phase": None,
            "chords": None,
        },
        {
            "case": "control_k4_genus3",
            "kind": "control",
            "embedding_family": "spatial_control",
            "graph": build_k4_genus3(),
            "wl_hash": None,
            "n_cycle_nodes": None,
            "n_chords": None,
            "phase": None,
            "chords": None,
        },
    ]


def checkpoint_accepted_cases(cases: list[dict], *, force: bool = False) -> None:
    """Persist accepted embeddings every CHECKPOINT_EVERY accepted cases."""
    if not SAVE_ACCEPTED_EMBEDDINGS or not cases:
        return

    should_save = force or (len(cases) % CHECKPOINT_EVERY == 0)
    if not should_save:
        return

    archive_path, manifest_path = save_embedding_archive(cases)

    print(
        f"CHECKPOINT SAVED: {len(cases)} accepted cases -> "
        f"{archive_path.name}"
    )



def generate_certified_ground_truths(
    count: int,
    *,
    seed: int,
) -> tuple[list[dict], dict]:
    """
    Generate EXACTLY `count` certified valid thick inputs.

    CROSSING_RICH_FRACTION is enforced on ACCEPTED random cases.
    Rejected candidates do not consume an index or a family quota.
    """
    if count < 3:
        raise ValueError(
            "count must be >= 3 because the three controls are retained."
        )

    rng = random.Random(seed)
    accepted: list[dict] = []
    accepted_random_hashes: set[str] = set()

    n_random_target = count - 3
    crossing_target = int(
        math.ceil(CROSSING_RICH_FRACTION * n_random_target)
    )
    baseline_target = n_random_target - crossing_target

    family_accepted = {
        "baseline": 0,
        "crossing_rich": 0,
    }

    rejection_counts = {
        "candidate_structure": 0,
        "duplicate_wl_hash": 0,
        "tube_under_resolved": 0,
        "input_topology_certificate_failed": 0,
        "crossing_tubes_too_close": 0,
        "reference_projection_failed": 0,
        "designed_crossings_not_verified": 0,
        "too_few_designed_crossings": 0,
        "other_certification_failure": 0,
    }

    # First certify the three canonical controls.
    for control in _control_cases():
        control.setdefault("designed_crossings", 0)
        control.setdefault("crossing_events", [])
        control.setdefault("min_designed_crossing_gap_world", None)

        valid, cert_control = certify_candidate(control)

        if not valid:
            raise RuntimeError(
                f"Canonical control {control['case']} failed certification: "
                f"{cert_control}"
            )

        if REQUIRE_MIN_THICK_RADIUS and not cert_control["thick_enough"]:
            raise RuntimeError(
                f"Canonical control {control['case']} is not thick enough: "
                f"{cert_control['radius_vox']:.3f} vox"
            )

        control["certification"] = cert_control
        accepted.append(control)
        checkpoint_accepted_cases(accepted)

        print(
            f"[accepted {len(accepted):4d}/{count}] "
            f"{control['case']:24s} "
            f"family={control['embedding_family']:24s} "
            f"g={graph_cycle_rank(control['graph']):2d} "
            f"r={cert_control['radius_vox']:5.2f} vox"
        )

    attempts = 0

    while len(accepted) < count:
        attempts += 1

        if attempts > MAX_GENERATION_ATTEMPTS:
            raise RuntimeError(
                f"Only {len(accepted)} certified cases were found after "
                f"{attempts} random candidate attempts. "
                f"Family progress={family_accepted}, "
                f"targets={{'baseline': {baseline_target}, "
                f"'crossing_rich': {crossing_target}}}. "
                "Increase MAX_GENERATION_ATTEMPTS or relax geometric constraints."
            )

        baseline_deficit = baseline_target - family_accepted["baseline"]
        crossing_deficit = crossing_target - family_accepted["crossing_rich"]

        if baseline_deficit <= 0:
            family = "crossing_rich"
        elif crossing_deficit <= 0:
            family = "baseline"
        else:
            # Pick in proportion to the remaining accepted-case deficit.
            total_deficit = baseline_deficit + crossing_deficit
            family = (
                "crossing_rich"
                if rng.random() < crossing_deficit / total_deficit
                else "baseline"
            )

        candidate = make_random_candidate(
            rng,
            embedding_family=family,
        )

        if not candidate.get("accepted_as_candidate", False):
            rejection_counts["candidate_structure"] += 1
            continue

        wl_hash = candidate["wl_hash"]

        if wl_hash in accepted_random_hashes:
            rejection_counts["duplicate_wl_hash"] += 1
            continue

        try:
            valid, cert_candidate = certify_candidate(candidate)
        except Exception:
            rejection_counts["other_certification_failure"] += 1
            continue

        if not valid:
            reason = cert_candidate.get(
                "reject_reason",
                "other_certification_failure",
            )
            if reason in rejection_counts:
                rejection_counts[reason] += 1
            else:
                rejection_counts["other_certification_failure"] += 1
            continue

        if (
            REQUIRE_MIN_THICK_RADIUS
            and not cert_candidate["thick_enough"]
        ):
            rejection_counts["tube_under_resolved"] += 1
            continue

        random_index = len(accepted) - 3
        candidate["case"] = f"random_{random_index:04d}"
        candidate.pop("accepted_as_candidate", None)
        candidate["certification"] = cert_candidate

        accepted_random_hashes.add(wl_hash)
        family_accepted[family] += 1
        accepted.append(candidate)
        checkpoint_accepted_cases(accepted)

        print(
            f"[accepted {len(accepted):4d}/{count}] "
            f"{candidate['case']:24s} "
            f"family={family:13s} "
            f"g={graph_cycle_rank(candidate['graph']):2d} "
            f"x={candidate['designed_crossings']:2d} "
            f"r={cert_candidate['radius_vox']:5.2f} vox "
            f"(attempts={attempts})"
        )

    diagnostics = {
        "requested_valid_cases": int(count),
        "accepted_valid_cases": int(len(accepted)),
        "random_candidate_attempts": int(attempts),
        "random_family_targets": {
            "baseline": int(baseline_target),
            "crossing_rich": int(crossing_target),
        },
        "random_family_accepted": {
            key: int(value)
            for key, value in family_accepted.items()
        },
        "rejection_counts": rejection_counts,
    }

    checkpoint_accepted_cases(accepted, force=True)
    return accepted, diagnostics


CASES, GENERATION_DIAGNOSTICS = generate_certified_ground_truths(
    N_GROUND_TRUTH,
    seed=RANDOM_SEED,
)

assert len(CASES) == N_GROUND_TRUTH
assert all(case["certification"]["input_topology_pass"] for case in CASES)
if REQUIRE_MIN_THICK_RADIUS:
    assert all(case["certification"]["thick_enough"] for case in CASES)

_random_cases = [
    case for case in CASES
    if case["kind"] == "random_cycle_chord"
]
_crossing_random_cases = [
    case for case in _random_cases
    if case.get("embedding_family") == "crossing_rich"
]
_required_crossing_random = int(
    math.ceil(CROSSING_RICH_FRACTION * len(_random_cases))
)
assert len(_crossing_random_cases) >= _required_crossing_random
assert all(
    case["designed_crossings"] >= MIN_DESIGNED_CROSSINGS
    for case in _crossing_random_cases
)

# Pairwise WL distinctness among accepted random abstract graphs.
accepted_random_hashes = [
    case["wl_hash"]
    for case in CASES
    if case["kind"] == "random_cycle_chord"
]
assert len(accepted_random_hashes) == len(set(accepted_random_hashes))

# The theta pair intentionally shares the same abstract MultiGraph.
assert nx.is_isomorphic(
    nx.MultiGraph(CASES[0]["graph"]),
    nx.MultiGraph(CASES[1]["graph"]),
)

print("\\nGeneration diagnostics:")
print(json.dumps(GENERATION_DIAGNOSTICS, indent=2))

if SAVE_ACCEPTED_EMBEDDINGS:
    EMBEDDING_ARCHIVE_PATH, EMBEDDING_MANIFEST_PATH = save_embedding_archive(CASES)
    print("\\nSaved exact accepted embeddings:")
    print("  archive :", EMBEDDING_ARCHIVE_PATH)
    print("  manifest:", EMBEDDING_MANIFEST_PATH)


# Summary of the ACTUAL accepted benchmark population.
ground_truth_summary = []

for case in CASES:
    stats = graph_stats(case["graph"])
    cert = case["certification"]

    ground_truth_summary.append(
        {
            "case": case["case"],
            "kind": case["kind"],
            "embedding_family": case.get("embedding_family", "control"),
            "designed_crossings": int(case.get("designed_crossings", 0)),
            "reference_projection_crossings": cert.get(
                "reference_projection_crossings"
            ),
            "crossing_gap_in_tube_radii": cert.get(
                "crossing_gap_in_tube_radii"
            ),
            "nodes": stats["nodes"],
            "edges": stats["edges"],
            "genus_cycle_rank": stats["cycle_rank"],
            "max_degree": stats["max_degree"],
            "radius_vox": cert["radius_vox"],
            "beta0": cert["observed_beta0"],
            "beta1": cert["observed_beta1"],
            "beta2": cert["observed_beta2"],
            "input_topology_pass": cert["input_topology_pass"],
            "thick_enough": cert["thick_enough"],
        }
    )

ground_truth_summary = pd.DataFrame(ground_truth_summary)
display(ground_truth_summary.head(20))

print("\\nCertified benchmark cases:", len(ground_truth_summary))
print(
    "Genus range:",
    int(ground_truth_summary["genus_cycle_rank"].min()),
    "to",
    int(ground_truth_summary["genus_cycle_rank"].max()),
)
print(
    "Minimum accepted radius [vox]:",
    float(ground_truth_summary["radius_vox"].min()),
)
print(
    "All inputs topology-certified:",
    bool(ground_truth_summary["input_topology_pass"].all()),
)


[accepted    1/1000] control_trivial_theta    family=baseline_control         g= 2 r= 9.95 vox
[accepted    2/1000] control_trefoil_theta    family=knotted_crossing_control g= 2 r=10.72 vox
[accepted    3/1000] control_k4_genus3        family=spatial_control          g= 3 r= 7.51 vox


/Users/hakanakgun/.venvs/nh-kg/lib/python3.13/site-packages/networkx/algorithms/graph_hashing.py:211: UserWarning: The hashes produced for graphs without node or edge attributes changed in v3.5 due to a bugfix (see documentation).
  node_labels = _init_node_labels(G, edge_attr, node_attr)


[accepted    4/1000] random_0000              family=baseline      g= 3 x= 0 r= 4.10 vox (attempts=2)
[accepted    5/1000] random_0001              family=baseline      g= 4 x= 0 r= 4.57 vox (attempts=25)
[accepted    6/1000] random_0002              family=crossing_rich g= 5 x= 4 r= 6.60 vox (attempts=31)
[accepted    7/1000] random_0003              family=baseline      g= 4 x= 0 r= 4.11 vox (attempts=36)
[accepted    8/1000] random_0004              family=baseline      g= 4 x= 0 r= 4.68 vox (attempts=38)
[accepted    9/1000] random_0005              family=crossing_rich g= 6 x= 6 r= 5.22 vox (attempts=61)
[accepted   10/1000] random_0006              family=baseline      g= 3 x= 0 r= 5.62 vox (attempts=67)
[accepted   11/1000] random_0007              family=baseline      g= 4 x= 0 r= 4.04 vox (attempts=69)
[accepted   12/1000] random_0008              family=baseline      g= 3 x= 0 r= 6.14 vox (attempts=70)
[accepted   13/1000] random_0009              family=baseline      g= 3 x=

KeyboardInterrupt: 


## 7. Reload an accepted embedding later

The compressed archive contains the exact node positions and edge polylines. Therefore a saved case can be reconstructed without rerunning the random generator.

Example:

```python
graph, record = reconstruct_saved_case("random_0000")
radius = record["certification"]["radius_world"]

volume, origin, spacing, _ = voxelize_regular_neighborhood(
    graph,
    radius,
    record_archive_grid_size,
)
```

For this notebook's archive, `record_archive_grid_size` is `GRID_SIZE`. The saved `certification` block also contains the original `origin_world`, `spacing_world`, `radius_vox`, and Betti numbers.


In [ ]:

# Example archive integrity check.
if SAVE_ACCEPTED_EMBEDDINGS:
    _archive_check = load_embedding_archive(EMBEDDING_ARCHIVE_PATH)
    assert len(_archive_check["cases"]) == N_GROUND_TRUTH

    _graph_check, _record_check = reconstruct_saved_case(
        _archive_check["cases"][0]["case"],
        EMBEDDING_ARCHIVE_PATH,
    )

    assert graph_stats(_graph_check)["cycle_rank"] == (
        _record_check["graph_stats"]["cycle_rank"]
    )

    print(
        "Archive reload check passed for",
        _record_check["case"],
    )



## 7. Production skeletonization and graph extraction


In [ ]:

def graph_voxel_to_world(
    graph: nx.MultiGraph,
    origin: np.ndarray,
    spacing: float,
) -> nx.MultiGraph:
    out = nx.MultiGraph(graph)

    for _, data in out.nodes(data=True):
        data["pos"] = (
            origin
            + spacing
            * np.asarray(
                data["pos"],
                dtype=float,
            )
        )

    for _, _, _, data in out.edges(
        keys=True,
        data=True,
    ):
        data["pts"] = (
            origin
            + spacing
            * np.asarray(
                data["pts"],
                dtype=float,
            )
        )

    return ensure_embedding(out)


def recover_graph_from_volume(
    volume: np.ndarray,
    *,
    origin: np.ndarray,
    spacing: float,
):
    """
    Recover and clean one spatial graph while timing every major substage.

    Timings use time.perf_counter() and exclude checkpoint/file-I/O overhead.
    """
    recovery_start = time.perf_counter()

    t = time.perf_counter()
    skeleton = skeletonize_volume(volume)
    skeleton_seconds = time.perf_counter() - t

    t = time.perf_counter()
    raw = topology_aware_skeleton_image_to_graph(
        skeleton,
        max_junction_degree=MAX_JUNCTION_DEGREE,
        adaptive_max_hops=ADAPTIVE_MAX_HOPS,
        anomaly_ratio=ANOMALY_RATIO,
    )
    extraction_seconds = time.perf_counter() - t

    t = time.perf_counter()
    raw_world = graph_voxel_to_world(
        raw,
        origin,
        spacing,
    )
    voxel_to_world_seconds = time.perf_counter() - t

    cleaned = raw_world.copy()

    t = time.perf_counter()
    cleaned = contract_short_edges(
        cleaned,
        min_length=CONTRACT_SHORT_EDGE_VOX * spacing,
        copy=False,
    )
    contract_short_edges_seconds = time.perf_counter() - t

    t = time.perf_counter()
    cleaned = remove_leaf_nodes(cleaned)
    remove_leaf_nodes_seconds = time.perf_counter() - t

    t = time.perf_counter()
    cleaned = simplify_edges(cleaned)
    simplify_edges_seconds = time.perf_counter() - t

    t = time.perf_counter()
    cleaned = smooth_edges(
        cleaned,
        epsilon=SMOOTH_EPSILON_VOX * spacing,
        copy=False,
    )
    smooth_edges_seconds = time.perf_counter() - t

    t = time.perf_counter()
    cleaned = ensure_embedding(cleaned)
    final_ensure_embedding_seconds = time.perf_counter() - t

    cleanup_seconds = (
        contract_short_edges_seconds
        + remove_leaf_nodes_seconds
        + simplify_edges_seconds
        + smooth_edges_seconds
        + final_ensure_embedding_seconds
    )

    recovery_total_seconds = time.perf_counter() - recovery_start

    return {
        "skeleton": skeleton,
        "raw_graph": raw_world,
        "graph": cleaned,
        "skeleton_seconds": float(skeleton_seconds),
        "extraction_seconds": float(extraction_seconds),
        "voxel_to_world_seconds": float(voxel_to_world_seconds),
        "contract_short_edges_seconds": float(contract_short_edges_seconds),
        "remove_leaf_nodes_seconds": float(remove_leaf_nodes_seconds),
        "simplify_edges_seconds": float(simplify_edges_seconds),
        "smooth_edges_seconds": float(smooth_edges_seconds),
        "final_ensure_embedding_seconds": float(final_ensure_embedding_seconds),
        "cleanup_seconds": float(cleanup_seconds),
        "recovery_total_seconds": float(recovery_total_seconds),
    }



## 8. Yamada preservation

For every subcubic case for which the production Yamada computation succeeds, the notebook compares

\[
Y(G_i)
\quad\text{with}\quad
Y(G_i^{\rm rec}).
\]

This is the strongest embedding-sensitive diagnostic used by this benchmark, but equality is not asserted to be a complete classification theorem for all spatial graphs.


In [ ]:

def canonical_polynomial(expression):
    if expression is None:
        return None

    return sp.expand(
        sp.cancel(expression)
    )


def polynomials_equal(a, b):
    if a is None or b is None:
        return None

    try:
        return bool(
            sp.simplify(a - b) == 0
        )
    except Exception:
        return bool(
            sp.expand(a) == sp.expand(b)
        )


def safe_yamada(graph: nx.MultiGraph):
    """
    Return (normalized_polynomial, error, elapsed_seconds).

    Elapsed time includes projection selection and Yamada evaluation.
    """
    t0 = time.perf_counter()

    if not COMPUTE_YAMADA:
        return None, "disabled", float(time.perf_counter() - t0)

    if not native_available():
        return (
            None,
            "native backend unavailable: " + str(native_import_error()),
            float(time.perf_counter() - t0),
        )

    stats = graph_stats(graph)

    if not stats["subcubic"]:
        return (
            None,
            f"max degree is {stats['max_degree']} (>3)",
            float(time.perf_counter() - t0),
        )

    try:
        result = compute_yamada_polynomial(
            graph,
            A,
            rotation_angles=None,
            num_rotation_samples=YAMADA_NUM_ROTATION_SAMPLES,
            normalize=True,
            n_jobs=YAMADA_N_JOBS,
            return_result=True,
        )

        return (
            canonical_polynomial(result.polynomial),
            None,
            float(time.perf_counter() - t0),
        )

    except Exception as exc:
        return (
            None,
            f"{type(exc).__name__}: {exc}",
            float(time.perf_counter() - t0),
        )


# ---------------------------------------------------------------------------
# Exact recovered-graph serialization
# ---------------------------------------------------------------------------

def graph_to_serializable_exact(graph: nx.MultiGraph) -> dict:
    """
    Serialize the recovered graph WITHOUT geometrically normalizing it.

    This preserves the post-cleanup node positions and edge polylines that
    were passed directly to safe_yamada().
    """
    nodes = []

    for node, data in graph.nodes(data=True):
        nodes.append(
            {
                "id": _json_node_id(node),
                "pos": np.asarray(data["pos"], dtype=float).tolist(),
            }
        )

    edges = []

    for u, v, key, data in graph.edges(keys=True, data=True):
        edges.append(
            {
                "u": _json_node_id(u),
                "v": _json_node_id(v),
                "key": _json_node_id(key),
                "pts": np.asarray(data["pts"], dtype=float).tolist(),
            }
        )

    return {
        "nodes": nodes,
        "edges": edges,
    }


def graph_payload_sha256(payload: dict) -> str:
    """Stable fingerprint of one serialized recovered spatial graph."""
    blob = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(blob).hexdigest()


## 10. Run the recovery framework, save exact recovered graphs, and benchmark runtime

For every successful recovery this section saves the **exact cleaned recovered
spatial graph** and the Yamada polynomial computed from that same graph.

It records per-case wall-clock runtimes using `time.perf_counter()` for:

1. thick-volume voxelization;
2. independent Betti/input-topology checking;
3. skeletonization;
4. skeleton-to-graph extraction;
5. voxel-to-world conversion;
6. short-edge contraction;
7. leaf removal;
8. edge simplification;
9. edge smoothing;
10. final embedding normalization;
11. total graph cleanup;
12. total volume-to-graph recovery;
13. abstract-isomorphism diagnostic;
14. ground-truth Yamada;
15. recovered-graph Yamada;
16. exact recovered-graph serialization;
17. graph fingerprinting;
18. volume-to-Yamada runtime;
19. synthetic end-to-end runtime;
20. total per-case runtime excluding checkpoint/file-I/O overhead.

Timing outputs are checkpointed every `CHECKPOINT_EVERY` tested cases to

```text
/Users/hakanakgun/Desktop/Projects/ProfLeeProjects/Knotted_graph_code_paper/
KnottedGraph_1earlier/User_guide/benchmarks/results/
```

as:

```text
04_thick_handlebody_pipeline_timings.csv
04_thick_handlebody_pipeline_timing_summary.csv
```

The summary CSV reports \(N\), mean, median, standard deviation, minimum,
25th/75th percentiles, 95th percentile, and maximum for each timed stage.

Random candidate-search/generation time is intentionally excluded from these
framework-runtime numbers because it is benchmark-data generation rather than
the tested volume-to-spatial-graph/Yamada workflow.


In [ ]:

HANDLEBODY_DATA_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_results(
    rows: list[dict],
    *,
    force: bool = False,
) -> None:
    """Persist tabular recovery results every CHECKPOINT_EVERY tested cases."""
    if not rows:
        return

    should_save = force or (len(rows) % CHECKPOINT_EVERY == 0)

    if not should_save:
        return

    checkpoint_path = HANDLEBODY_DATA_DIR / RESULTS_CHECKPOINT_NAME
    tmp_path = checkpoint_path.with_suffix(
        checkpoint_path.suffix + ".tmp"
    )

    pd.DataFrame(rows).to_csv(
        tmp_path,
        index=False,
    )
    tmp_path.replace(
        checkpoint_path
    )

    print(
        f"RESULT CHECKPOINT SAVED: {len(rows)} tested cases -> "
        f"{checkpoint_path.name}"
    )


def checkpoint_recovered_graphs(
    recovered_records: list[dict],
    *,
    tested_count: int,
    force: bool = False,
) -> None:
    """
    Persist exact post-cleanup recovered graphs.

    Checkpoint cadence is based on TESTED cases, so it remains synchronized
    with the result CSV even when an individual recovery fails.
    """
    should_save = force or (
        tested_count > 0
        and tested_count % CHECKPOINT_EVERY == 0
    )

    if not should_save:
        return

    archive_path = HANDLEBODY_DATA_DIR / RECOVERED_GRAPH_ARCHIVE_NAME
    manifest_path = HANDLEBODY_DATA_DIR / RECOVERED_GRAPH_MANIFEST_NAME

    archive = {
        "schema_version": 1,
        "description": (
            "Exact post-cleanup recovered spatial graphs paired with the "
            "Yamada polynomial computed from that same graph."
        ),
        "git_branch": _git_text("branch", "--show-current"),
        "git_commit": _git_text("rev-parse", "HEAD"),
        "tested_count": int(tested_count),
        "checkpoint_every": int(CHECKPOINT_EVERY),
        "cases": recovered_records,
    }

    manifest_rows = []

    for record in recovered_records:
        stats = record["recovered_graph_stats"]

        manifest_rows.append(
            {
                "index": record["index"],
                "case": record["case"],
                "kind": record["kind"],
                "embedding_family": record["embedding_family"],
                "designed_crossings": record["designed_crossings"],
                "nodes": stats["nodes"],
                "edges": stats["edges"],
                "cycle_rank": stats["cycle_rank"],
                "max_degree": stats["max_degree"],
                "recovered_graph_sha256": record[
                    "recovered_graph_sha256"
                ],
                "recovered_yamada": record["recovered_yamada"],
                "recovered_yamada_error": record[
                    "recovered_yamada_error"
                ],
                "recovered_yamada_graph_sha256": record[
                    "recovered_yamada_graph_sha256"
                ],
                "true_yamada": record["true_yamada"],
                "yamada_match": record["yamada_match"],
                "primary_topology_pass": record[
                    "primary_topology_pass"
                ],
                "overall_pass": record["overall_pass"],
            }
        )

    archive_tmp = archive_path.with_suffix(
        archive_path.suffix + ".tmp"
    )
    manifest_tmp = manifest_path.with_suffix(
        manifest_path.suffix + ".tmp"
    )

    with gzip.open(
        archive_tmp,
        "wt",
        encoding="utf-8",
    ) as handle:
        json.dump(
            archive,
            handle,
            separators=(",", ":"),
            allow_nan=False,
        )

    pd.DataFrame(
        manifest_rows
    ).to_csv(
        manifest_tmp,
        index=False,
    )

    archive_tmp.replace(
        archive_path
    )
    manifest_tmp.replace(
        manifest_path
    )

    print(
        f"RECOVERED-GRAPH CHECKPOINT SAVED: "
        f"{len(recovered_records)} recovered graphs from "
        f"{tested_count} tested cases -> {archive_path.name}"
    )


def checkpoint_all(
    rows: list[dict],
    recovered_records: list[dict],
    *,
    force: bool = False,
) -> None:
    """Atomically checkpoint both benchmark tables and exact recovered graphs."""
    checkpoint_results(
        rows,
        force=force,
    )

    checkpoint_recovered_graphs(
        recovered_records,
        tested_count=len(rows),
        force=force,
    )


# ---------------------------------------------------------------------------
# Paper timing CSVs
# ---------------------------------------------------------------------------

TIMING_STAGE_COLUMNS = [
    "voxelization_seconds",
    "input_topology_check_seconds",
    "skeletonization_seconds",
    "graph_extraction_seconds",
    "voxel_to_world_seconds",
    "contract_short_edges_seconds",
    "remove_leaf_nodes_seconds",
    "simplify_edges_seconds",
    "smooth_edges_seconds",
    "final_ensure_embedding_seconds",
    "graph_cleanup_seconds",
    "recovery_total_seconds",
    "abstract_isomorphism_seconds",
    "true_yamada_seconds",
    "recovered_yamada_seconds",
    "yamada_total_seconds",
    "serialize_recovered_graph_seconds",
    "fingerprint_seconds",
    "volume_to_yamada_seconds",
    "synthetic_end_to_end_seconds",
    "case_total_seconds",
]


def _timing_summary_frame(
    timing_rows: list[dict],
) -> pd.DataFrame:
    frame = pd.DataFrame(timing_rows)
    summary_rows = []

    for stage in TIMING_STAGE_COLUMNS:
        if stage not in frame.columns:
            continue

        values = pd.to_numeric(
            frame[stage],
            errors="coerce",
        ).dropna()

        values = values[
            np.isfinite(values)
            & (values >= 0)
        ]

        if len(values) == 0:
            continue

        summary_rows.append(
            {
                "stage": stage,
                "n": int(len(values)),
                "mean_seconds": float(values.mean()),
                "median_seconds": float(values.median()),
                "std_seconds": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
                "min_seconds": float(values.min()),
                "p25_seconds": float(values.quantile(0.25)),
                "p75_seconds": float(values.quantile(0.75)),
                "p95_seconds": float(values.quantile(0.95)),
                "max_seconds": float(values.max()),
            }
        )

    return pd.DataFrame(summary_rows)


def checkpoint_timings(
    timing_rows: list[dict],
    *,
    force: bool = False,
) -> None:
    if not timing_rows:
        return

    should_save = (
        force
        or len(timing_rows) % CHECKPOINT_EVERY == 0
    )

    if not should_save:
        return

    TIMING_RESULTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    timing_path = (
        TIMING_RESULTS_DIR
        / TIMING_RESULTS_CSV_NAME
    )
    summary_path = (
        TIMING_RESULTS_DIR
        / TIMING_SUMMARY_CSV_NAME
    )

    timing_tmp = timing_path.with_suffix(
        timing_path.suffix + ".tmp"
    )
    summary_tmp = summary_path.with_suffix(
        summary_path.suffix + ".tmp"
    )

    pd.DataFrame(
        timing_rows
    ).to_csv(
        timing_tmp,
        index=False,
    )

    _timing_summary_frame(
        timing_rows
    ).to_csv(
        summary_tmp,
        index=False,
    )

    timing_tmp.replace(timing_path)
    summary_tmp.replace(summary_path)

    print(
        f"TIMING CHECKPOINT SAVED: {len(timing_rows)} tested cases -> "
        f"{timing_path}"
    )
    print(
        f"TIMING SUMMARY UPDATED -> {summary_path}"
    )


def checkpoint_everything(
    rows: list[dict],
    recovered_records: list[dict],
    timing_rows: list[dict],
    *,
    force: bool = False,
) -> None:
    checkpoint_all(
        rows,
        recovered_records,
        force=force,
    )

    checkpoint_timings(
        timing_rows,
        force=force,
    )


rows = []
recovered_records = []
timing_rows = []
REPRESENTATIVE = None

run_git_branch = _git_text("branch", "--show-current")
run_git_commit = _git_text("rev-parse", "HEAD")
run_platform = platform.platform()
run_processor = platform.processor()
run_python = platform.python_version()

for index, case in enumerate(CASES):
    case_name = case["case"]
    seed = case["graph"]
    cert = case["certification"]

    seed_stats = graph_stats(seed)
    expected_betti = expected_handlebody_betti(seed)

    timing_row = {
        "index": int(index),
        "case": case_name,
        "kind": case["kind"],
        "embedding_family": case.get("embedding_family", "control"),
        "designed_crossings": int(case.get("designed_crossings", 0)),
        "grid_size": int(GRID_SIZE),
        "expected_genus": int(seed_stats["cycle_rank"]),
        "seed_nodes": int(seed_stats["nodes"]),
        "seed_edges": int(seed_stats["edges"]),
        "radius_vox": None,
        "completed": False,
        "overall_pass": False,
        "timing_error": None,
        "git_branch": run_git_branch,
        "git_commit": run_git_commit,
        "python_version": run_python,
        "platform": run_platform,
        "processor": run_processor,
    }

    for _stage in TIMING_STAGE_COLUMNS:
        timing_row[_stage] = None

    case_start = time.perf_counter()

    try:
        radius_limit = float(cert["radius_limit_world"])
        radius = float(cert["radius_world"])

        # 1. Spatial graph -> thick binary volume
        t = time.perf_counter()
        volume, origin, spacing, half_extent = (
            voxelize_regular_neighborhood(
                seed,
                radius,
                GRID_SIZE,
            )
        )
        voxelization_seconds = time.perf_counter() - t
        timing_row["voxelization_seconds"] = float(voxelization_seconds)

        radius_vox = radius / spacing
        timing_row["radius_vox"] = float(radius_vox)

        # 2. Independent digital topology check
        t = time.perf_counter()
        betti = volume_betti(volume)
        input_topology_check_seconds = time.perf_counter() - t
        timing_row["input_topology_check_seconds"] = float(
            input_topology_check_seconds
        )

        input_topology_pass = betti == expected_betti
        thick_enough = radius_vox >= MIN_THICK_RADIUS_VOX

        if not input_topology_pass:
            raise RuntimeError(
                f"Certified case {case_name} failed deterministic topology "
                f"recheck: expected={expected_betti}, observed={betti}"
            )

        if REQUIRE_MIN_THICK_RADIUS and not thick_enough:
            raise RuntimeError(
                f"Certified case {case_name} failed thickness recheck: "
                f"{radius_vox:.3f} < {MIN_THICK_RADIUS_VOX:.3f} vox"
            )

        if not np.allclose(
            origin,
            np.asarray(cert["origin_world"], dtype=float),
        ):
            raise RuntimeError(
                f"Certified case {case_name} changed voxel origin."
            )

        if not np.isclose(
            spacing,
            float(cert["spacing_world"]),
        ):
            raise RuntimeError(
                f"Certified case {case_name} changed voxel spacing."
            )

        base = {
            "index": index,
            "case": case_name,
            "kind": case["kind"],
            "embedding_family": case.get("embedding_family", "control"),
            "designed_crossings": int(case.get("designed_crossings", 0)),
            "reference_projection_crossings": cert.get(
                "reference_projection_crossings"
            ),
            "crossing_gap_in_tube_radii": cert.get(
                "crossing_gap_in_tube_radii"
            ),
            "wl_hash": case.get("wl_hash"),
            "grid_size": GRID_SIZE,
            "thickness_fraction": THICKNESS_FRACTION,
            "radius_limit_world": radius_limit,
            "radius_world": radius,
            "spacing_world": float(spacing),
            "radius_vox": float(radius_vox),
            "thick_enough": bool(thick_enough),
            "occupied_voxels": int(np.count_nonzero(volume)),
            "occupancy_fraction": float(np.mean(volume)),
            "voxelization_seconds": float(voxelization_seconds),
            "input_topology_check_seconds": float(input_topology_check_seconds),
            "seed_nodes": seed_stats["nodes"],
            "seed_edges": seed_stats["edges"],
            "expected_genus": seed_stats["cycle_rank"],
            "volume_beta0": int(betti["beta0"]),
            "volume_beta1": int(betti["beta1"]),
            "volume_beta2": int(betti["beta2"]),
            "volume_chi": int(betti["chi"]),
            "input_topology_pass": True,
        }

        # 3. Binary volume -> cleaned spatial graph
        recovered = recover_graph_from_volume(
            volume,
            origin=origin,
            spacing=spacing,
        )

        timing_row["skeletonization_seconds"] = recovered["skeleton_seconds"]
        timing_row["graph_extraction_seconds"] = recovered["extraction_seconds"]
        timing_row["voxel_to_world_seconds"] = recovered["voxel_to_world_seconds"]
        timing_row["contract_short_edges_seconds"] = recovered[
            "contract_short_edges_seconds"
        ]
        timing_row["remove_leaf_nodes_seconds"] = recovered[
            "remove_leaf_nodes_seconds"
        ]
        timing_row["simplify_edges_seconds"] = recovered["simplify_edges_seconds"]
        timing_row["smooth_edges_seconds"] = recovered["smooth_edges_seconds"]
        timing_row["final_ensure_embedding_seconds"] = recovered[
            "final_ensure_embedding_seconds"
        ]
        timing_row["graph_cleanup_seconds"] = recovered["cleanup_seconds"]
        timing_row["recovery_total_seconds"] = recovered["recovery_total_seconds"]

        raw_stats = graph_stats(recovered["raw_graph"])
        clean_stats = graph_stats(recovered["graph"])

        cycle_rank_match = (
            clean_stats["cycle_rank"]
            == seed_stats["cycle_rank"]
        )

        # 4. Abstract-graph diagnostic
        t = time.perf_counter()
        abstract_isomorphic = nx.is_isomorphic(
            nx.MultiGraph(seed),
            nx.MultiGraph(recovered["graph"]),
        )
        timing_row["abstract_isomorphism_seconds"] = float(
            time.perf_counter() - t
        )

        # 5. Ground-truth Yamada
        (
            true_yamada,
            true_yamada_error,
            true_yamada_seconds,
        ) = safe_yamada(seed)
        timing_row["true_yamada_seconds"] = float(true_yamada_seconds)

        # 6. Recovered-graph Yamada
        (
            recovered_yamada,
            recovered_yamada_error,
            recovered_yamada_seconds,
        ) = safe_yamada(recovered["graph"])
        timing_row["recovered_yamada_seconds"] = float(
            recovered_yamada_seconds
        )
        timing_row["yamada_total_seconds"] = float(
            true_yamada_seconds
            + recovered_yamada_seconds
        )

        yamada_match = polynomials_equal(
            true_yamada,
            recovered_yamada,
        )

        primary_topology_pass = bool(
            clean_stats["connected"]
            and clean_stats["subcubic"]
            and cycle_rank_match
        )

        embedding_pass = primary_topology_pass

        if yamada_match is not None:
            embedding_pass = embedding_pass and bool(yamada_match)

        overall_pass = bool(
            input_topology_pass
            and (thick_enough or not REQUIRE_MIN_THICK_RADIUS)
            and embedding_pass
        )

        # 7. Exact recovered-graph serialization
        t = time.perf_counter()
        recovered_payload = graph_to_serializable_exact(
            recovered["graph"]
        )
        timing_row["serialize_recovered_graph_seconds"] = float(
            time.perf_counter() - t
        )

        # 8. Fingerprint
        t = time.perf_counter()
        recovered_graph_sha256 = graph_payload_sha256(
            recovered_payload
        )
        timing_row["fingerprint_seconds"] = float(
            time.perf_counter() - t
        )

        # Useful manuscript aggregates.
        timing_row["volume_to_yamada_seconds"] = float(
            recovered["recovery_total_seconds"]
            + recovered_yamada_seconds
        )

        timing_row["synthetic_end_to_end_seconds"] = float(
            voxelization_seconds
            + input_topology_check_seconds
            + recovered["recovery_total_seconds"]
            + recovered_yamada_seconds
        )

        recovered_record = {
            "index": int(index),
            "case": case_name,
            "kind": case["kind"],
            "embedding_family": case.get("embedding_family", "control"),
            "designed_crossings": int(case.get("designed_crossings", 0)),
            "grid_size": int(GRID_SIZE),
            "radius_world": float(radius),
            "spacing_world": float(spacing),
            "recovered_graph": recovered_payload,
            "recovered_graph_sha256": recovered_graph_sha256,
            "recovered_graph_stats": clean_stats,
            "recovered_yamada": (
                None
                if recovered_yamada is None
                else str(recovered_yamada)
            ),
            "recovered_yamada_error": recovered_yamada_error,
            "recovered_yamada_graph_sha256": (
                recovered_graph_sha256
                if recovered_yamada is not None
                else None
            ),
            "true_yamada": (
                None
                if true_yamada is None
                else str(true_yamada)
            ),
            "true_yamada_error": true_yamada_error,
            "yamada_match": yamada_match,
            "primary_topology_pass": bool(primary_topology_pass),
            "overall_pass": bool(overall_pass),
        }

        recovered_records.append(recovered_record)

        # Complete timer before any checkpoint/file I/O.
        timing_row["case_total_seconds"] = float(
            time.perf_counter() - case_start
        )
        timing_row["completed"] = True
        timing_row["overall_pass"] = bool(overall_pass)

        rows.append(
            {
                **base,
                **{
                    f"raw_{key}": value
                    for key, value in raw_stats.items()
                },
                **{
                    f"recovered_{key}": value
                    for key, value in clean_stats.items()
                },
                "recovered_graph_sha256": recovered_graph_sha256,
                "skeleton_voxels": int(
                    np.count_nonzero(recovered["skeleton"])
                ),
                "skeleton_seconds": recovered["skeleton_seconds"],
                "extraction_seconds": recovered["extraction_seconds"],
                "voxel_to_world_seconds": recovered["voxel_to_world_seconds"],
                "contract_short_edges_seconds": recovered[
                    "contract_short_edges_seconds"
                ],
                "remove_leaf_nodes_seconds": recovered[
                    "remove_leaf_nodes_seconds"
                ],
                "simplify_edges_seconds": recovered["simplify_edges_seconds"],
                "smooth_edges_seconds": recovered["smooth_edges_seconds"],
                "final_ensure_embedding_seconds": recovered[
                    "final_ensure_embedding_seconds"
                ],
                "cleanup_seconds": recovered["cleanup_seconds"],
                "recovery_total_seconds": recovered["recovery_total_seconds"],
                "abstract_isomorphism_seconds": timing_row[
                    "abstract_isomorphism_seconds"
                ],
                "cycle_rank_match": bool(cycle_rank_match),
                "abstract_isomorphic": bool(abstract_isomorphic),
                "true_yamada": (
                    None
                    if true_yamada is None
                    else str(true_yamada)
                ),
                "true_yamada_error": true_yamada_error,
                "true_yamada_seconds": float(true_yamada_seconds),
                "recovered_yamada": (
                    None
                    if recovered_yamada is None
                    else str(recovered_yamada)
                ),
                "recovered_yamada_error": recovered_yamada_error,
                "recovered_yamada_seconds": float(
                    recovered_yamada_seconds
                ),
                "recovered_yamada_graph_sha256": (
                    recovered_graph_sha256
                    if recovered_yamada is not None
                    else None
                ),
                "yamada_match": yamada_match,
                "primary_topology_pass": primary_topology_pass,
                "overall_pass": overall_pass,
                "recovery_error": None,
            }
        )

        timing_rows.append(timing_row)

        checkpoint_everything(
            rows,
            recovered_records,
            timing_rows,
        )

        if (
            REPRESENTATIVE is None
            and case["kind"] == "random_cycle_chord"
            and seed_stats["cycle_rank"] >= 4
        ):
            REPRESENTATIVE = {
                "case": case_name,
                "volume": volume.copy(),
                "origin": origin.copy(),
                "spacing": spacing,
                "graph": recovered["graph"].copy(),
                "radius_vox": radius_vox,
            }

        print(
            f"[{index+1:4d}/{N_GROUND_TRUTH}] "
            f"{case_name:24s} "
            f"{'PASS' if overall_pass else 'FAIL':4s} "
            f"{case.get('embedding_family','control'):13s} "
            f"g={seed_stats['cycle_rank']:2d} "
            f"x={int(case.get('designed_crossings',0)):2d} "
            f"r={radius_vox:5.2f} vox "
            f"Y={yamada_match} "
            f"graph={recovered_graph_sha256[:10]} "
            f"T={timing_row['case_total_seconds']:.3f}s"
        )

    except Exception as exc:
        timing_row["case_total_seconds"] = float(
            time.perf_counter() - case_start
        )
        timing_row["timing_error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        rows.append(
            {
                "index": index,
                "case": case_name,
                "kind": case["kind"],
                "embedding_family": case.get("embedding_family", "control"),
                "designed_crossings": int(case.get("designed_crossings", 0)),
                "grid_size": GRID_SIZE,
                "thickness_fraction": THICKNESS_FRACTION,
                "input_topology_pass": True,
                "recovery_error": (
                    f"{type(exc).__name__}: {exc}"
                ),
                "overall_pass": False,
            }
        )

        timing_rows.append(timing_row)

        checkpoint_everything(
            rows,
            recovered_records,
            timing_rows,
        )

        print(
            f"[{index+1:4d}/{N_GROUND_TRUTH}] "
            f"{case_name:24s} ERROR: "
            f"{type(exc).__name__}: {exc} "
            f"T={timing_row['case_total_seconds']:.3f}s"
        )

checkpoint_everything(
    rows,
    recovered_records,
    timing_rows,
    force=True,
)

results = pd.DataFrame(rows)
timing_results = pd.DataFrame(timing_rows)
timing_summary = _timing_summary_frame(timing_rows)

assert len(results) == N_GROUND_TRUTH
assert len(timing_results) == N_GROUND_TRUTH
assert len(CASES) == N_GROUND_TRUTH

for record in recovered_records:
    if record["recovered_yamada"] is not None:
        assert (
            record["recovered_yamada_graph_sha256"]
            == record["recovered_graph_sha256"]
        )

display(results.head(25))

print("\\nPer-case timing preview:")
display(timing_results.head(25))

print("\\nTiming summary for paper reporting:")
display(timing_summary)

if SAVE_RESULTS_CSV:
    output_csv = HANDLEBODY_DATA_DIR / RESULTS_CSV_NAME
    results.to_csv(output_csv, index=False)
    print("Saved:", output_csv)

print(
    "Exact recovered graphs saved:",
    len(recovered_records),
    "->",
    HANDLEBODY_DATA_DIR / RECOVERED_GRAPH_ARCHIVE_NAME,
)

print(
    "Per-case timing CSV:",
    TIMING_RESULTS_DIR / TIMING_RESULTS_CSV_NAME,
)

print(
    "Timing summary CSV:",
    TIMING_RESULTS_DIR / TIMING_SUMMARY_CSV_NAME,
)


### Exact recovered-graph archive integrity check

The fingerprint check below verifies that the serialized graph still matches
the fingerprint stored beside its recovered Yamada value.


In [ ]:

RECOVERED_GRAPH_ARCHIVE_PATH = (
    ROOT
    / RECOVERED_GRAPH_ARCHIVE_NAME
)

with gzip.open(
    RECOVERED_GRAPH_ARCHIVE_PATH,
    "rt",
    encoding="utf-8",
) as handle:
    _recovered_archive_check = (
        json.load(handle)
    )

for _record in _recovered_archive_check[
    "cases"
]:
    _actual_hash = graph_payload_sha256(
        _record["recovered_graph"]
    )

    assert (
        _actual_hash
        == _record[
            "recovered_graph_sha256"
        ]
    )

    if (
        _record[
            "recovered_yamada"
        ]
        is not None
    ):
        assert (
            _record[
                "recovered_yamada_graph_sha256"
            ]
            == _actual_hash
        )

print(
    "Recovered archive integrity passed:",
    len(
        _recovered_archive_check[
            "cases"
        ]
    ),
    "exact recovered graphs",
)


## Timing outputs for paper reporting

Use the per-case CSV for scaling plots and case-level error bars. Use the
summary CSV for compact manuscript runtime statements. Each per-case timing
row also stores grid size, genus, graph size, tube radius, pass/completion
status, git commit, Python version, and platform metadata.


In [ ]:

TIMING_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timing_csv_path = (
    TIMING_RESULTS_DIR
    / TIMING_RESULTS_CSV_NAME
)
timing_summary_path = (
    TIMING_RESULTS_DIR
    / TIMING_SUMMARY_CSV_NAME
)

print("Per-case timing CSV:", timing_csv_path)
print("Timing summary CSV:", timing_summary_path)

if timing_csv_path.exists():
    _timing_saved = pd.read_csv(timing_csv_path)
    print("Saved timing rows:", len(_timing_saved))

if timing_summary_path.exists():
    _summary_saved = pd.read_csv(timing_summary_path)
    display(_summary_saved)


## 11. Aggregate result


In [ ]:

def _mean_boolean(column):
    if column not in results.columns:
        return np.nan

    return float(
        pd.to_numeric(
            results[column],
            errors="coerce",
        ).mean()
    )


print("============================================================")
print(f"{N_GROUND_TRUTH}-CASE CERTIFIED THICK-HANDLEBODY VALIDATION")
print("============================================================")
print("Cases:", len(results))
print("Grid size:", GRID_SIZE)
print("Thickness fraction:", THICKNESS_FRACTION)
print()
print(
    "Input topology certificate pass rate:",
    _mean_boolean("input_topology_pass"),
)
print(
    "Cycle-rank recovery rate:",
    _mean_boolean("cycle_rank_match"),
)
print(
    "Abstract graph isomorphism rate:",
    _mean_boolean("abstract_isomorphic"),
)
print(
    "Yamada preservation rate:",
    _mean_boolean("yamada_match"),
)
print(
    "Overall pass rate:",
    _mean_boolean("overall_pass"),
)

if "radius_vox" in results.columns:
    valid_radius = pd.to_numeric(
        results["radius_vox"],
        errors="coerce",
    )

    print()
    print(
        "Median tube radius [vox]:",
        float(valid_radius.median()),
    )
    print(
        "Minimum tube radius [vox]:",
        float(valid_radius.min()),
    )
    print(
        "Maximum tube radius [vox]:",
        float(valid_radius.max()),
    )

summary_columns = [
    "case",
    "kind",
    "expected_genus",
    "radius_vox",
    "input_topology_pass",
    "cycle_rank_match",
    "abstract_isomorphic",
    "yamada_match",
    "overall_pass",
    "recovery_error",
]

summary_columns = [
    column
    for column in summary_columns
    if column in results.columns
]

failures = results.loc[
    ~results["overall_pass"].fillna(False)
].copy()

print()
print("Number of failures:", len(failures))

if len(failures):
    display(
        failures[summary_columns].head(100)
    )

if FAIL_ON_RECOVERY_FAILURE and len(failures):
    raise AssertionError(
        f"{len(failures)} of "
        f"{N_GROUND_TRUTH} benchmark cases failed."
    )



## 12. Genus-resolved recovery success rates

Because **all inputs were certified before the framework was called**, these curves now measure recovery behavior rather than contamination from invalid voxelized ground truths.


In [ ]:

if "expected_genus" in results.columns:
    genus_summary = (
        results.dropna(
            subset=["expected_genus"]
        )
        .groupby("expected_genus")
        .agg(
            n_cases=("case", "size"),
            median_radius_vox=("radius_vox", "median"),
            input_pass_rate=("input_topology_pass", "mean"),
            cycle_rank_pass_rate=("cycle_rank_match", "mean"),
            yamada_pass_rate=("yamada_match", "mean"),
            overall_pass_rate=("overall_pass", "mean"),
        )
        .reset_index()
    )

    display(genus_summary)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(
        genus_summary["expected_genus"],
        genus_summary["overall_pass_rate"],
        marker="o",
    )
    ax.set_xlabel("Ground-truth handlebody genus")
    ax.set_ylabel("Recovery success rate")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("Thick-handlebody recovery versus genus")
    plt.tight_layout()
    plt.show()



## Crossing-resolved recovery success rates

`designed_crossings` counts deliberately constructed over/under events in the reference \(xy\) projection. These are spatial crossings, not graph vertices: the two centerlines have a finite \(z\)-gap and the certified tube neighborhoods remain disjoint.

This table makes it possible to ask whether recovery degrades as the number of genuine tubular over/under events increases.


In [ ]:

crossing_rows = results[
    results.get("embedding_family", pd.Series(index=results.index, dtype=object))
    .eq("crossing_rich")
].copy()

if len(crossing_rows):
    crossing_summary = (
        crossing_rows.groupby("designed_crossings")
        .agg(
            n_cases=("case", "size"),
            median_radius_vox=("radius_vox", "median"),
            median_crossing_gap_radii=(
                "crossing_gap_in_tube_radii",
                "median",
            ),
            cycle_rank_pass_rate=("cycle_rank_match", "mean"),
            yamada_pass_rate=("yamada_match", "mean"),
            overall_pass_rate=("overall_pass", "mean"),
        )
        .reset_index()
    )

    display(crossing_summary)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(
        crossing_summary["designed_crossings"],
        crossing_summary["overall_pass_rate"],
        marker="o",
    )
    ax.set_xlabel("Designed over/under crossings")
    ax.set_ylabel("Recovery success rate")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title("Recovery versus tubular crossing complexity")
    plt.tight_layout()
    plt.show()
else:
    print("No crossing-rich rows were present.")



## 13. Representative thick object and recovered spine


In [ ]:

if REPRESENTATIVE is None:
    print(
        "No representative recovered case was available."
    )
else:
    rep = REPRESENTATIVE

    vertices, faces, _, _ = marching_cubes(
        rep["volume"].astype(np.float32),
        level=0.5,
        spacing=(
            rep["spacing"],
            rep["spacing"],
            rep["spacing"],
        ),
        step_size=2,
    )

    vertices = (
        vertices
        + rep["origin"]
    )

    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(
        111,
        projection="3d",
    )

    ax.plot_trisurf(
        vertices[:, 0],
        vertices[:, 1],
        faces,
        vertices[:, 2],
        linewidth=0.0,
        alpha=0.22,
    )

    for _, _, _, data in rep["graph"].edges(
        keys=True,
        data=True,
    ):
        points = np.asarray(
            data["pts"],
            dtype=float,
        )

        ax.plot(
            points[:, 0],
            points[:, 1],
            points[:, 2],
            linewidth=2.0,
        )

    ax.set_title(
        f"{rep['case']}: recovered graph inside thick handlebody "
        f"(r={rep['radius_vox']:.1f} vox)"
    )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_box_aspect((1, 1, 1))

    plt.tight_layout()
    plt.show()



## Interpretation

`N_GROUND_TRUTH` is now the exact number of **valid thick handlebodies actually tested**.

The generator may attempt many more candidates, but a candidate does not enter the benchmark unless:

1. its voxelized regular neighborhood has exactly the intended handlebody topology;
2. it satisfies the configured minimum voxel thickness (when enabled);
3. its accepted random abstract topology is WL-distinct from the other accepted random cases.

Therefore an `overall_pass=False` row now refers to a failure **after a valid input was supplied to the recovery framework**, rather than a failure of synthetic-input construction.

The compressed embedding archive is the permanent ground-truth data product. It stores enough information to reconstruct every accepted spatial graph and regenerate its thick regular-neighborhood volume later.
